# 08 - Tempo de prova de volta ao dataset

Esta etapa le `csvs/07_dataset_transformado.csv` e traz de volta o tempo total de prova de cada piloto, reconstruido a partir do arquivo bruto `csvs/01_dataset_completo.csv`. O arquivo de entrada nao e alterado; o resultado sai em `csvs/08_dataset_com_tempo_de_prova.csv`, com as mesmas linhas e duas colunas a mais:

| Coluna | Conteudo |
|---|---|
| `rd_race_time_millis` | Tempo total de prova em milissegundos (original ou reconstruido). |
| `rd_race_time_fonte` | De onde veio o tempo: `original`, `vencedor + gap`, `vencedor + voltas x melhor volta` ou vazio. |

**Atencao:** o tempo de prova e resultado da propria corrida (vazamento, secao 2 de `02_Analise_Dados.ipynb`). Ele nao entra como feature para prever a ordem antes da largada; serve como alvo alternativo ou para analise.

In [1]:
import pandas as pd
from IPython.display import display

ENTRADA = 'csvs/07_dataset_transformado.csv'
BRUTO = 'csvs/01_dataset_completo.csv'
SAIDA = 'csvs/08_dataset_com_tempo_de_prova.csv'
CHAVE = ['race_id', 'driver_id']

df = pd.read_csv(ENTRADA, low_memory=False)
bruto = pd.read_csv(BRUTO, low_memory=False)
print(f'Linhas: {len(df):,} | Colunas: {df.shape[1]} | Linhas no bruto: {len(bruto):,}')

Linhas: 8,462 | Colunas: 51 | Linhas no bruto: 9,146


## 1. O que existe no bruto

No bruto, `rd_race_time_millis` ja e o tempo total (o do vencedor somado ao gap), nao a diferenca. Para quem nao tem tempo, ha tres situacoes:

- **Gap em milissegundos** (`rd_race_gap_millis`): terminou na mesma volta do vencedor.
- **Gap em voltas** (`rd_race_gap_laps`): terminou com uma ou mais voltas de atraso (`+1 lap`).
- **Nada**: abandonou, foi desclassificado ou nao largou.

In [2]:
tem_tempo = bruto['rd_race_time_millis'].notna()
tem_gap = bruto['rd_race_gap_millis'].notna()
tem_voltas = bruto['rd_race_gap_laps'].notna()
tem_melhor_volta = bruto['sess_fastest_lap_time_millis'].notna()

display(pd.crosstab(
    [tem_tempo.rename('tem tempo'), tem_gap.rename('tem gap (ms)')],
    tem_voltas.rename('tem gap (voltas)'),
))
print(f'Gap em voltas com melhor volta do piloto: {(tem_voltas & tem_melhor_volta).sum():,} de {tem_voltas.sum():,}')

tem gap (voltas)        False  True 
tem tempo tem gap (ms)              
False     False          1887   2744
True      False           436      0
          True           4079      0

Gap em voltas com melhor volta do piloto: 2,744 de 2,744


## 2. Reconstrucao

Em ordem de prioridade:

1. **Original:** `rd_race_time_millis` quando existe.
2. **Vencedor + gap:** tempo do vencedor + `rd_race_gap_millis`.
3. **Vencedor + voltas x melhor volta:** tempo do vencedor + `rd_race_gap_laps` x melhor volta do proprio piloto naquela corrida (`sess_fastest_lap_time_millis`). E uma estimativa: a melhor volta e mais rapida que uma volta media, entao o tempo sai um pouco otimista.
4. **Sem informacao:** fica vazio (`NaN`), por enquanto.

In [3]:
ORIGINAL = 'original'
GAP = 'vencedor + gap'
VOLTAS = 'vencedor + voltas x melhor volta'

vencedor = bruto['rd_position_display_order'].eq(1)
assert vencedor.groupby(bruto['race_id']).sum().eq(1).all()
assert bruto.loc[vencedor, 'rd_race_time_millis'].notna().all()
tempo_vencedor = bruto['race_id'].map(bruto.loc[vencedor].set_index('race_id')['rd_race_time_millis'])

por_gap = tempo_vencedor + bruto['rd_race_gap_millis']
por_voltas = tempo_vencedor + bruto['rd_race_gap_laps'] * bruto['sess_fastest_lap_time_millis']

tempos = bruto[CHAVE].copy()
tempos['rd_race_time_millis'] = bruto['rd_race_time_millis'].combine_first(por_gap).combine_first(por_voltas)
tempos['rd_race_time_fonte'] = pd.Series(pd.NA, index=bruto.index, dtype='string')
tempos.loc[por_voltas.notna(), 'rd_race_time_fonte'] = VOLTAS
tempos.loc[por_gap.notna(), 'rd_race_time_fonte'] = GAP
tempos.loc[tem_tempo, 'rd_race_time_fonte'] = ORIGINAL

assert tempos.duplicated(CHAVE).sum() == 0
assert tempos['rd_race_time_millis'].notna().eq(tempos['rd_race_time_fonte'].notna()).all()
display(tempos['rd_race_time_fonte'].value_counts(dropna=False).rename('linhas no bruto').to_frame())

,linhas no bruto
rd_race_time_fonte,
original,4515
vencedor + voltas x melhor volta,2744
<NA>,1887


## 3. Conferencias

- Onde o bruto ja tinha tempo, o valor nao muda.
- Quem tem gap em ms ja tem tempo no bruto: a regra 2 confere com o original (diferencas pequenas vem de penalidades de tempo).
- Dentro de cada corrida, conferimos se o tempo respeita a ordem final. A regra 3 quebra a ordem em parte dos casos, quase sempre entre dois pilotos com o mesmo numero de voltas de atraso: quem chegou atras mas tem melhor volta mais rapida recebe um tempo menor. E limitacao da estimativa, nao erro de juncao.

In [4]:
assert tempos.loc[tem_tempo, 'rd_race_time_millis'].equals(bruto.loc[tem_tempo, 'rd_race_time_millis'])

diferenca_gap = (por_gap - bruto['rd_race_time_millis']).abs().dropna()
print(f'Regra 2 vs original: {len(diferenca_gap):,} linhas | diferenca maxima {diferenca_gap.max():,.0f} ms | linhas diferentes {(diferenca_gap > 0).sum()}')

com_tempo = tempos.assign(ordem=bruto['rd_position_display_order']).dropna(subset=['rd_race_time_millis'])
com_tempo = com_tempo.sort_values(['race_id', 'ordem'])
fora_de_ordem = com_tempo.groupby('race_id')['rd_race_time_millis'].diff().lt(0)
print(f'Pares consecutivos fora de ordem: {fora_de_ordem.sum()} de {len(com_tempo):,} linhas com tempo')

Regra 2 vs original: 4,079 linhas | diferenca maxima 526 ms | linhas diferentes 18
Pares consecutivos fora de ordem: 956 de 7,259 linhas com tempo


## 4. Juntando ao dataset

O tempo entra por `race_id` + `driver_id`. O arquivo 07 ja nao tem as corridas encurtadas (`03`), entao algumas linhas do bruto ficam de fora.

In [5]:
df_com_tempo = df.merge(tempos, on=CHAVE, how='left', validate='one_to_one')

assert len(df_com_tempo) == len(df)
assert df_com_tempo[df.columns].equals(df)

resumo = df_com_tempo['rd_race_time_fonte'].value_counts(dropna=False).rename('linhas').to_frame()
resumo['%'] = resumo['linhas'].div(len(df_com_tempo)).mul(100).round(1)
display(resumo)

df_com_tempo.to_csv(SAIDA, index=False)
print(f'Colunas: {df.shape[1]} -> {df_com_tempo.shape[1]}')
print(f'Arquivo salvo: {SAIDA}')

,linhas,%
rd_race_time_fonte,,
original,4154,49.1
vencedor + voltas x melhor volta,2581,30.5
<NA>,1727,20.4


Colunas: 51 -> 53
Arquivo salvo: csvs/08_dataset_com_tempo_de_prova.csv
